# RAG System with ChromaDB and OpenAI

This notebook implements a Retrieval Augmented Generation (RAG) system that:
1. Reads documents from a folder
2. Splits them into chunks
3. Stores embeddings in ChromaDB
4. Retrieves relevant chunks for queries
5. Generates answers using GPT-3.5

## Cell 1: Install Requirements

In [1]:
!pip install -r requirements.txt -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wheel 0.46.3 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.


## Cell 2: Import Libraries

In [1]:
import os
from pathlib import Path
from typing import List

import chromadb
from chromadb.config import Settings
from openai import OpenAI

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    UnstructuredFileLoader
)

print("All libraries imported successfully!")

/Users/udaybijjala/miniconda3/envs/myenv_rag/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


All libraries imported successfully!


## Cell 3: Set OpenAI API Key

In [2]:
# Set your OpenAI API key here
OPENAI_API_KEY = "API_KEY"  # Replace with your actual API key

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Model configurations
EMBEDDING_MODEL = "text-embedding-ada-002"  # OpenAI Ada embedding model
GENERATION_MODEL = "gpt-3.5-turbo"  # GPT-3.5 for answer generation

print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Generation Model: {GENERATION_MODEL}")

Embedding Model: text-embedding-ada-002
Generation Model: gpt-3.5-turbo


## Cell 4: Initialize ChromaDB

In [3]:
# Initialize ChromaDB with persistent storage
CHROMA_DB_PATH = "./chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Create or get the collection
COLLECTION_NAME = "document_chunks"

# Delete existing collection if it exists (for fresh start)
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except:
    pass

# Create new collection
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity for distance
)

print(f"ChromaDB collection '{COLLECTION_NAME}' created successfully!")
print(f"Database path: {CHROMA_DB_PATH}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


ChromaDB collection 'document_chunks' created successfully!
Database path: ./chroma_db


## Cell 5: Document Loading Functions

In [4]:
DOCUMENTS_FOLDER = "./documents"

def load_document(file_path: str):
    """Load a document based on its file extension."""
    file_extension = Path(file_path).suffix.lower()
    
    try:
        if file_extension == ".pdf":
            print(f"Loading PDF: {file_path}")
            loader = PyPDFLoader(file_path)
        elif file_extension == ".txt":
            print(f"Loading TXT: {file_path}")
            loader = TextLoader(file_path, encoding="utf-8")
        elif file_extension in [".docx", ".doc"]:
            print(f"Loading DOCX: {file_path}")
            loader = Docx2txtLoader(file_path)
        else:
            loader = UnstructuredFileLoader(file_path)
        
        return loader.load()
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


def load_all_documents(folder_path: str):
    """Load all documents from the specified folder."""
    documents = []
    folder = Path(folder_path)
    
    if not folder.exists():
        print(f"Folder '{folder_path}' does not exist!")
        return documents
    
    supported_extensions = [".pdf", ".txt", ".docx", ".doc", ".md"]
    
    for file_path in folder.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in supported_extensions:
            print(f"Loading: {file_path.name}")
            docs = load_document(str(file_path))
            
            # Add document name to metadata
            for doc in docs:
                doc.metadata["document_name"] = file_path.name
            
            documents.extend(docs)
    
    print(f"\nTotal documents loaded: {len(documents)}")
    return documents

print("Document loading functions defined.")

Document loading functions defined.


## Cell 6: Text Splitting Function

In [5]:
def split_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    chunks = text_splitter.split_documents(documents)
    print(f"Documents split into {len(chunks)} chunks")
    return chunks

print("Text splitting function defined.")

Text splitting function defined.


## Cell 7: Embedding Generation Function

In [6]:
def generate_embedding(text: str) -> List[float]:
    """Generate embedding for a text using OpenAI Ada model."""
    response = client.embeddings.create(
        input=text,
        model=EMBEDDING_MODEL
    )
    return response.data[0].embedding


def generate_embeddings_batch(texts: List[str]) -> List[List[float]]:
    """Generate embeddings for multiple texts in batch."""
    response = client.embeddings.create(
        input=texts,
        model=EMBEDDING_MODEL
    )
    return [item.embedding for item in response.data]

print("Embedding generation functions defined.")

Embedding generation functions defined.


## Cell 8: Store Chunks in ChromaDB

In [7]:
def store_chunks_in_chromadb(chunks, batch_size: int = 100):
    """Store document chunks with embeddings in ChromaDB."""
    total_chunks = len(chunks)
    
    for i in range(0, total_chunks, batch_size):
        batch = chunks[i:i + batch_size]
        
        ids = [f"chunk_{i + j}" for j in range(len(batch))]
        texts = [chunk.page_content for chunk in batch]
        metadatas = [
            {
                "document_name": chunk.metadata.get("document_name", "unknown"),
                "page": chunk.metadata.get("page", 0),
                "chunk_index": i + j
            }
            for j, chunk in enumerate(batch)
        ]
        
        print(f"Generating embeddings for chunks {i} to {i + len(batch) - 1}...")
        embeddings = generate_embeddings_batch(texts)
        
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        
        print(f"Stored {i + len(batch)}/{total_chunks} chunks")
    
    print(f"\nAll {total_chunks} chunks stored in ChromaDB!")

print("ChromaDB storage function defined.")

ChromaDB storage function defined.


## Cell 9: Load, Split, and Store Documents

In [8]:
# Load all documents from the documents folder
print("=" * 50)
print("STEP 1: Loading Documents")
print("=" * 50)
documents = load_all_documents(DOCUMENTS_FOLDER)

if documents:
    print("\n" + "=" * 50)
    print("STEP 2: Splitting Documents into Chunks")
    print("=" * 50)
    chunks = split_documents(documents, chunk_size=1000, chunk_overlap=200)
    
    print("\n" + "=" * 50)
    print("STEP 3: Generating Embeddings and Storing in ChromaDB")
    print("=" * 50)
    store_chunks_in_chromadb(chunks)
    
    print("\n" + "=" * 50)
    print("Document processing complete!")
    print("=" * 50)
else:
    print("\nNo documents found. Add documents to:", os.path.abspath(DOCUMENTS_FOLDER))

STEP 1: Loading Documents
Loading: RFP SM-PRC0001034 Addendum 1.pdf
Loading PDF: documents/RFP SM-PRC0001034 Addendum 1.pdf


/Users/udaybijjala/miniconda3/envs/myenv_rag/lib/python3.10/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Loading: Scope responses for ID Verification.pdf
Loading PDF: documents/Scope responses for ID Verification.pdf
Loading: Alamo College RFP V1.2.pdf
Loading PDF: documents/Alamo College RFP V1.2.pdf
Loading: RFP_Som-AJKEditsv1.pdf
Loading PDF: documents/RFP_Som-AJKEditsv1.pdf

Total documents loaded: 41

STEP 2: Splitting Documents into Chunks
Documents split into 61 chunks

STEP 3: Generating Embeddings and Storing in ChromaDB
Generating embeddings for chunks 0 to 60...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored 61/61 chunks

All 61 chunks stored in ChromaDB!

Document processing complete!


## Cell 10: Query Functions - Retrieve Similar Chunks

In [9]:
def retrieve_similar_chunks(query: str, top_k: int = 5):
    """Retrieve top-k similar chunks for a query."""
    print(f"Generating embedding for query: '{query}'")
    query_embedding = generate_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    return results


def display_retrieved_chunks(results):
    """Display the retrieved chunks with metadata and distances."""
    print("\n" + "=" * 60)
    print("RETRIEVED CHUNKS (Top 5)")
    print("=" * 60)
    
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]
    
    for i, (doc, metadata, distance) in enumerate(zip(documents, metadatas, distances), 1):
        print(f"\n--- Chunk {i} ---")
        print(f"Document: {metadata['document_name']}")
        print(f"Distance (Cosine): {distance:.4f}")
        print(f"Similarity Score: {1 - distance:.4f}")
        print(f"Content Preview: {doc[:300]}..." if len(doc) > 300 else f"Content: {doc}")
    
    print("\n" + "=" * 60)

print("Query functions defined.")

Query functions defined.


## Cell 11: Answer Generation Function

In [10]:
def generate_answer(query: str, context_chunks: List[str], metadata_list: List[dict]) -> str:
    """Generate an answer using GPT-3.5 based on retrieved context."""
    
    context_with_sources = ""
    for i, (chunk, metadata) in enumerate(zip(context_chunks, metadata_list), 1):
        context_with_sources += f"\n[Source {i}: {metadata['document_name']}]\n{chunk}\n"
    
    system_prompt = """You are a helpful assistant that answers questions based on the provided context. 
    Always base your answers on the given context. If the context doesn't contain enough information 
    to answer the question, say so. When possible, cite the source document for your information."""
    
    user_prompt = f"""Context:
{context_with_sources}

Question: {query}

Please provide a comprehensive answer based on the context above. Cite the source documents when relevant."""
    
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=1000
    )
    
    return response.choices[0].message.content

print("Answer generation function defined.")

Answer generation function defined.


## Cell 12: Complete RAG Query Pipeline

In [11]:
def rag_query(query: str, top_k: int = 5, show_chunks: bool = True) -> str:
    """Complete RAG pipeline: retrieve relevant chunks and generate answer."""
    
    print(f"\nQuery: {query}")
    print("-" * 60)
    
    print("\nStep 1: Retrieving relevant chunks from ChromaDB...")
    results = retrieve_similar_chunks(query, top_k=top_k)
    
    if show_chunks:
        display_retrieved_chunks(results)
    
    print("\nStep 2: Generating answer using GPT-3.5...")
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    
    answer = generate_answer(query, documents, metadatas)
    
    print("\n" + "=" * 60)
    print("GENERATED ANSWER")
    print("=" * 60)
    print(answer)
    print("=" * 60)
    
    return answer

print("RAG query pipeline defined.")

RAG query pipeline defined.


## Cell 13: Test the RAG System

In [12]:
# Example query - replace with your own question
user_query = "What is the main topic of the documents?"

# Run the RAG query
answer = rag_query(user_query, top_k=5, show_chunks=True)


Query: What is the main topic of the documents?
------------------------------------------------------------

Step 1: Retrieving relevant chunks from ChromaDB...
Generating embedding for query: 'What is the main topic of the documents?'


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



RETRIEVED CHUNKS (Top 5)

--- Chunk 1 ---
Document: Alamo College RFP V1.2.pdf
Distance (Cosine): 0.2410
Similarity Score: 0.7590
Content Preview: d) Provide  efforts  the firm makes  to keep its professionals  informed  of 
developments relevant to the industry.  
3) Extent  to Which  the Goods  or Services  Meet  the District’s  Needs:  
a) A brief discussion of your firm’s background and experience in providing the  
requested goods and ser...

--- Chunk 2 ---
Document: Alamo College RFP V1.2.pdf
Distance (Cosine): 0.2443
Similarity Score: 0.7557
Content Preview: ExamRoom.AI  Page  | 2  
 
 
CONTENTS  
 
 
MINIMUM  QUALIFICATIONS  ................................ ................................ ................................ ................................ ........  3 
SECTION 1  SCOPE  OF WORK ................................ ..............................

--- Chunk 3 ---
Document: RFP SM-PRC0001034 Addendum 1.pdf
Distance (Cosine): 0.2455
Similarity Score: 0.7545
Content Pre

## Cell 14: Interactive Query Mode

In [ ]:
def interactive_query():
    """Run queries interactively in a loop."""
    print("=" * 60)
    print("INTERACTIVE RAG QUERY MODE")
    print("Type 'quit' or 'exit' to stop")
    print("=" * 60)
    
    while True:
        query = input("\nEnter your question: ").strip()
        
        if query.lower() in ["quit", "exit", "q"]:
            print("\nGoodbye!")
            break
        
        if not query:
            print("Please enter a valid question.")
            continue
        
        rag_query(query, top_k=5, show_chunks=False)

# Uncomment the line below to start interactive mode
# interactive_query()

## Cell 15: Utility Functions

In [ ]:
def get_collection_stats():
    """Display statistics about the ChromaDB collection."""
    count = collection.count()
    print(f"Total chunks in collection: {count}")
    
    if count > 0:
        sample = collection.peek(limit=min(count, 10))
        unique_docs = set()
        for meta in sample["metadatas"]:
            unique_docs.add(meta.get("document_name", "unknown"))
        print(f"Sample documents: {unique_docs}")
    
    return count


def clear_collection():
    """Clear all data from the collection."""
    global collection
    chroma_client.delete_collection(name=COLLECTION_NAME)
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )
    print("Collection cleared and recreated.")


# Display current stats
get_collection_stats()